# General Summary

This script is designed for loading, inspecting, and potentially modifying a dataset of cellular features stored in a pickle file. It uses various Python libraries such as `pandas` for data handling, `matplotlib` and `seaborn` for visualization, and `scipy.stats` for statistical analysis. A predefined list of feature names outlines specific metrics in the dataset, such as area, intensity, and region-level characteristics. The code includes optional steps for replacing missing values, modifying specific features, and adjusting display settings. The primary goal is to prepare and explore the dataset for further analysis.


In [1]:
# Import necessary libraries
import pandas as pd  # For data manipulation and analysis
import pickle  # For working with serialized Python objects
import numpy as np  # For numerical computations
import matplotlib.pyplot as plt  # For creating visualizations
from scipy.stats import norm, lognorm  # For statistical distributions
import scipy.stats as stats  # For various statistical functions
import seaborn as sns  # For enhanced data visualizations

# Define a list of feature names to work with later in the dataset
FeatureNameList = [
    "Feature Area", "Feature Intensity",
    "All Children Count", "Child 1 Count", "Child 2 Count",
    "Region:Circularity(Mean)", "Region:Clumpiness(Mean)",
    "Region:Diameter, Max(Mean)", "Region:Diameter, Mean(Mean)",
    "Region:Diameter, Min(Mean)", "Region:Heterogeneity(Mean)",
    "Region:Hole Area(Mean)", "Region:Hole Area Ratio(Mean)",
    "Region:Roundness(Mean)", "Child 1 Intensity (mean)",
    "Child 2 Intensity (mean)"
]

# Load a dataset from a pickle file containing pre-processed data
plate_3_zsn2 = pd.read_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_1_zs.pkl')

# Uncomment the following line if you want to replace NaN values with 0
# plate_3_zsn2 = plate_3_zsn2.fillna(0)

# Uncomment the following line to display all rows of the dataset when printed
# pd.set_option('display.max_rows', plate_3_zsn2.shape[0] + 1)

# Uncomment and adjust the code below if you need to add a specific value (e.g., 1) to certain features
# value_to_add = 1

# Add the specified value to all columns listed in FeatureNameList
# plate_3_zsn2.loc[:, FeatureNameList] += value_to_add

# Print the entire DataFrame to visually inspect its contents
print(plate_3_zsn2)


       Display Name  Unnamed: 0
0               B02           0
1               B02           1
2               B02           2
3               B02           3
4               B02           4
...             ...         ...
658610          G11      658610
658611          G11      658611
658612          G11      658612
658613          G11      658613
658614          G11      658614

[658615 rows x 2 columns]




This script focuses on loading and saving datasets for cellular feature analysis:
1. **CSV to DataFrame**: A dataset is loaded from a CSV file (`merged_p3_30k.csv`) into a pandas DataFrame.
2. **Save as Pickle**: The loaded DataFrame is converted and saved as a pickle file (`merged_p3_30k.pkl`) for faster future access.
3. **Inspect Columns**: The column names of the dataset are printed to examine its structure.
4. **Optional Analysis**: There is commented-out code to load and inspect another pickle file (`batch1_plate_3.pkl`) for additional data exploration.


In [2]:
# Load a dataset from a CSV file into a pandas DataFrame
topkl = pd.read_csv(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\merged_p3_30k.csv')

# Convert the DataFrame to a pickle file for faster loading in future operations
topkl1 = topkl.to_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\merged_p3_30k.pkl')

# Print the column names of the loaded DataFrame to inspect its structure
print(topkl.columns)

# The following lines are commented out and would load a previously saved pickle file for additional analysis
# dfp = pd.read_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\h441 redo excel test\batch1_plate_3.pkl')
# print(dfp)


Index(['Unnamed: 0', 'Collect#', 'Display Name', 'Feature Name',
       'Object:Class', 'Object:Class Name', 'Region:Circularity',
       'Region:Clumpiness', 'Region:Diameter, Mean',
       'Region:Fractal Dimension', 'Region:Heterogeneity', 'Region:Hole Area',
       'Region:Hole Area Ratio', 'Region:Roundness', 'Nuclear Area',
       'Percent Area Parent', 'Feature Area',
       'Region:MCA Intensity of Feature (mean)(Sum|Child 2)',
       'Region:MCA Intensity of Feature (mean)(Sum|Child 3)'],
      dtype='object')


This script is designed for processing and analyzing cellular feature data extracted from high-content imaging experiments. The main steps include:

1. **Data Loading**:
   - Loads pre-processed data from a pickle file (`merged_p1_30k.pkl`) into a pandas DataFrame.
   - Renames specific columns for ease of use (e.g., `red_intensity` and `green_intensity`).

2. **Data Filtering and Indexing**:
   - Sets the index to `Display Name` and filters the DataFrame to include only specific wells for analysis (e.g., wells `B03`, `C03`, etc.).

3. **Feature Manipulation**:
   - Adjusts selected features (`FeatureNameList`) by adding a constant value (`value_to_add = 1`).

4. **Log Transformation and Standardization**:
   - For each feature, performs log transformation and calculates Z-scores using the mean and standard deviation of the log-transformed data.

5. **Feature Extraction via Custom Function**:
   - Defines a `DataExtractor` function to extract, filter, and standardize data for specific wells based on features like `Feature Area`.

6. **Iterative Data Aggregation**:
   - Iterates through a predefined grid of well names (e.g., `B02`, `C03`, etc.) to:
     - Extract data for each well using the `DataExtractor` function.
     - Append extracted Z-scores and metadata (e.g., well name, feature values) into aggregated lists.

7. **Data Flattening and Preparation**:
   - Flattens the aggregated lists to create a single dataset.
   - Constructs a new DataFrame to store the Z-scores and associated metadata.

8. **Final Output**:
   - Prepares and optionally saves the processed dataset as a pickle file for future analysis.
   - Prints key summary statistics and checks, such as the lengths of the aggregated lists.

## Purpose
This script serves to preprocess, filter, and standardize cellular feature data for subsequent analyses. It automates the handling of well-level data, computes statistical metrics, and prepares the data for further modelling or visualization tasks.


In [3]:
# Import required libraries
import pandas as pd  # For data manipulation and analysis
import pickle  # For loading and saving serialized Python objects
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For plotting
from scipy.stats import norm, lognorm  # For statistical distributions
import scipy.stats as stats  # For statistical functions
import seaborn as sns  # For enhanced visualizations

# Load a pre-processed dataset from a pickle file
dfp = pd.read_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\merged_p1_30k.pkl')

# Rename columns for easier reference
dfp1x = dfp.rename(columns={
    "Region:MCA Intensity of Feature (mean)(Sum|Child 3)": "red_intensity",
    "Region:MCA Intensity of Feature (mean)(Sum|Child 2)": "green_intensity"
})

# Set 'Display Name' as the index and filter rows based on specific well names
dfp1x = dfp1x.set_index(['Display Name'])
dfp3 = dfp1x.loc[dfp1x.index.isin([
    'B03', 'C03', 'D03', 'E03', 'F03', 'G03'  # Selected wells
])]

# Reset the index of the filtered DataFrame (optional)
dfp3.reset_index()

# Define a list of features to work with
FeatureNameList = [
    'Region:Circularity', 'Region:Clumpiness', 'Region:Diameter, Mean',
    'Region:Fractal Dimension', 'Region:Heterogeneity',
    'Region:Hole Area Ratio', 'Region:Roundness', 'Nuclear Area',
    'Percent Area Parent', 'Feature Area', 'red_intensity', 'green_intensity'
]

# Add a constant value to all selected features
value_to_add = 1
dfp3.loc[:, FeatureNameList] += value_to_add

# Loop through each feature to compute log transformation and standardization
for p in range(len(FeatureNameList)):
    dfp4 = dfp3[FeatureNameList[p]]  # Select a single feature
    dfp44 = np.log(dfp4)  # Apply log transformation

    dfp1 = pd.DataFrame()  # Initialize an empty DataFrame for results

    # Compute the mean and standard deviation of the log-transformed feature
    mean = dfp44.mean()
    std = dfp44.std()

    # Define a function to extract and standardize data for a specific well
    def DataExtractor(Well, Feature=p):
        # Load well-specific data from an NPZ file
        Well_data_names = np.load("C:/Users/ak20adb/OneDrive - University of Hertfordshire/Desktop/Celleste 30k test/full datasets/plate_1/" + Well + ".npz")
        Well_data = Well_data_names['FeatureData']  # Feature data
        Well_names = Well_data_names['FeatureNames']  # Feature names

        # Filter data based on minimum feature area (optional, currently disabled)
        MinFeatureArea = 0
        GoodDataIndices = np.where(Well_data[:, 0])[0]  # Keep all rows
        Well_data = Well_data[GoodDataIndices, :]
        Well_names = Well_names[GoodDataIndices]

        # Add a constant to the selected feature and compute Z-scores
        Well_data[:, Feature] = Well_data[:, Feature] + 1
        zscore = (np.log(Well_data[:, Feature]) - mean) / std

        # Prepare output DataFrames for well metadata and feature counts
        df = pd.DataFrame({'wells': [Well]})
        df1 = pd.DataFrame({'count': [len(Well_names)]})
        df2 = pd.concat([df, df1], ignore_index=True)

        # Create lists for storing results
        well_list = [Well]
        zscore_list = [zscore]

        return well_list, zscore_list, Well_names

    # Initialize lists to store aggregated results
    appended_data = []
    appended_data_letter = []
    appended_data_number = []
    appended_data2 = []
    append_lst = []
    appended_try = []
    appended_try2 = []

    # Define letters and numbers representing well names
    letter = ['B', 'C', 'D', 'E', 'F', 'G']
    num = ['02', '03', '04', '05', '06', '07', '08', '09', '10', '11']

    # Iterate over all combinations of well letters and numbers
    for i in range(len(num)):
        r = str(num[i])
        for x in range(len(letter)):
            try:
                # Create well name and extract data
                lst = str(letter[x]) + str(num[i])
                dfx1, dfx2, dfx3 = DataExtractor(lst)

                # Append data to the respective lists
                for e in dfx3:
                    appended_try.append(lst)
                appended_data.append(dfx1)
                appended_data_letter.append(letter[x])
                appended_data_number.append(num[i])
                appended_data2.append(dfx2[0])
                append_lst.append(lst)

            except:
                # Handle missing data
                appended_data.append([lst])
                appended_data_letter.append(letter[x])
                appended_data_number.append(num[i])
                appended_data2.append({0})
                append_lst.append(lst)
                appended_try2.append(lst)

    # Flatten aggregated results into a single list
    flat_list = [item for sublist in appended_data2 for item in sublist]

    # Create a DataFrame from the aggregated results
    plate_3_zsn = pd.DataFrame(appended_try, columns=['Display Name'])

    # Save the processed DataFrame to a pickle file (optional)
    # plate_3_zsn.to_pickle('C:/Users/ak20adb/OneDrive - University of Hertfordshire/Desktop/Celleste 30k test/full datasets/plate_1_zs.pkl')

# Print summary statistics for validation
print("Length of lst:", lst)
print("Length of flat_list:", len(flat_list))
print("Length of appended_try:", len(appended_try))
print("Length of appended_data2:", len(appended_data2))
print("Length of append_lst:", len(append_lst))

# Print the final DataFrame
print(plate_3_zsn)


c:\Users\ak20adb\Anaconda3\lib\site-packages\pandas\core\indexing.py:1884: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, val, pi)


ValueError: Length of values (33270) does not match length of index (658615)


## Purpose
This script processes cellular feature data extracted from high-content imaging experiments. It performs the following key steps:
1. **Data Loading**: Loads pre-processed feature data from pickle and NPZ files.
2. **Data Transformation**: Renames columns, filters rows, and applies mathematical transformations (e.g., log transformation, Z-score normalization).
3. **Feature Analysis**: Aggregates and processes features for specified wells in a plate layout.
4. **Result Compilation**: Prepares a final dataset for further analysis or visualization.

---

## Key Steps

### 1. Importing Libraries
- The script uses `pandas`, `numpy`, `matplotlib`, `seaborn`, and `scipy` for data handling, transformation, statistical analysis, and visualization.

### 2. Data Loading and Renaming
- Loads a pickle file containing feature data (`merged_p1_30k.pkl`).
- Renames columns for easier reference:
  - `Region:MCA Intensity of Feature (mean)(Sum|Child 3)` → `red_intensity`
  - `Region:MCA Intensity of Feature (mean)(Sum|Child 2)` → `green_intensity`.

### 3. Filtering Data
- Filters the dataset to include only specific wells (`B03`, `C03`, `D03`, etc.).
- Sets `Display Name` as the index for easier filtering.

### 4. Feature Transformation
- Defines a list of features to analyze (`FeatureNameList`) including metrics like `Region:Circularity`, `Region:Clumpiness`, `red_intensity`, and `green_intensity`.
- Adds a constant value (`1`) to all selected features for standardization purposes.
- Applies log transformation and computes Z-scores for each feature.

### 5. Well-Specific Data Extraction
- Defines a `DataExtractor` function to process data for individual wells:
  - Loads data from NPZ files.
  - Filters based on feature area (if necessary).
  - Computes Z-scores for the selected features.
- Aggregates results for each well in a predefined plate layout (rows: `B` to `G`, columns: `02` to `11`).

### 6. Aggregation and Flattening
- Collects and flattens Z-scores and feature values into a single list.
- Updates the DataFrame with the processed feature values.

### 7. Output and Validation
- Prints the lengths of key lists (e.g., `flat_list`) for validation.
- Displays the final DataFrame with processed feature data for the selected wells.

---

## Applications
This script is part of a pipeline for analyzing high-content imaging data. It prepares a standardized and processed dataset for:
- Visualization
- Statistical analysis
- Machine learning or modeling tasks


In [ ]:
# Import required libraries
import pandas as pd  # For data manipulation and analysis
import pickle  # For loading and saving serialized data
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For data visualization
from scipy.stats import norm, lognorm  # For statistical distributions
import scipy.stats as stats  # For statistical computations
import seaborn as sns  # For advanced visualizations

# Load a pre-processed dataset from a pickle file
dfp = pd.read_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\merged_p1_30k.pkl')

# Rename columns to more user-friendly names
dfp1x = dfp.rename(columns={
    "Region:MCA Intensity of Feature (mean)(Sum|Child 3)": "red_intensity",
    "Region:MCA Intensity of Feature (mean)(Sum|Child 2)": "green_intensity"
})

# Set 'Display Name' as the index for easier filtering and manipulation
dfp1x = dfp1x.set_index(['Display Name'])

# Filter the dataset to include only specified wells
dfp3 = dfp1x.loc[dfp1x.index.isin([
    'B03', 'C03', 'D03', 'E03', 'F03', 'G03'  # Selected wells
])]

# Reset the index of the filtered dataset (optional for further processing)
dfp3.reset_index()

# Define a list of features for analysis
FeatureNameList = [
    'Region:Circularity', 'Region:Clumpiness', 'Region:Diameter, Mean',
    'Region:Fractal Dimension', 'Region:Heterogeneity',
    'Region:Hole Area Ratio', 'Region:Roundness', 'Nuclear Area',
    'Percent Area Parent', 'Feature Area', 'red_intensity', 'green_intensity'
]

# Add a constant value (1) to all selected features
value_to_add = 1
dfp3.loc[:, FeatureNameList] += value_to_add

# Loop through each feature to compute transformations and statistics
for p in range(len(FeatureNameList)):
    # Select the current feature for processing
    dfp4 = dfp3[FeatureNameList[p]]
    # Apply log transformation to the feature
    dfp44 = np.log(dfp4)

    # Compute mean and standard deviation of the log-transformed feature
    mean = dfp44.mean()
    std = dfp44.std()

    # Define a function to extract data from specific wells and standardize it
    def DataExtractor(Well, Feature=p):
        # Load well-specific data from an NPZ file
        Well_data_names = np.load("C:/Users/ak20adb/OneDrive - University of Hertfordshire/Desktop/Celleste 30k test/full datasets/plate_1/" + Well + ".npz")
        Well_data = Well_data_names['FeatureData']  # Feature data
        Well_names = Well_data_names['FeatureNames']  # Feature names

        # Filter data based on minimum feature area (currently not applied)
        MinFeatureArea = 0
        GoodDataIndices = np.where(Well_data[:, 0])[0]
        Well_data = Well_data[GoodDataIndices, :]
        Well_names = Well_names[GoodDataIndices]

        # Add a constant to the feature and compute Z-scores
        Well_data[:, Feature] = Well_data[:, Feature] + 1
        zscore = (np.log(Well_data[:, Feature]) - mean) / std

        # Prepare outputs: well name, z-scores, and feature names
        well_list = [Well]
        zscore_list = [zscore]

        return well_list, zscore_list, Well_names

    # Prepare to store aggregated data
    appended_data = []
    appended_try = []
    appended_try2 = []
    letter = ['B', 'C', 'D', 'E', 'F', 'G']  # Row letters
    num = ['02', '03', '04', '05', '06', '07', '08', '09', '10', '11']  # Column numbers

    # Iterate through each well in the plate
    for i in range(len(num)):
        for x in range(len(letter)):
            try:
                # Construct well name and extract data
                lst = str(letter[x]) + str(num[i])
                dfx1, dfx2, dfx3 = DataExtractor(lst)

                # Append extracted data for processing
                appended_try.extend([lst] * len(dfx3))
                appended_data.append(dfx2[0])

            except:
                # Handle missing or problematic data
                appended_data.append([lst])
                appended_try.append(lst)

    # Flatten the list of results for final processing
    flat_list = [item for sublist in appended_data for item in sublist]

    # Update the DataFrame with processed feature values
    dfp1x = dfp1x.reset_index()
    plate_3_zs = dfp1x.iloc[:, [0, 1]]
    plate_3_zs[FeatureNameList[p]] = flat_list

# Print summary statistics to validate processing
print("Length of flat_list:", len(flat_list))
print("Length of appended_try:", len(appended_try))
print("Final DataFrame:")
print(plate_3_zs)


# Code Summary

## Purpose
This script processes cellular feature data from high-content imaging experiments. It performs data filtering, feature transformations, Z-score standardization, and aggregates results for specific wells in a plate layout.

---

## Key Steps

### 1. Import Libraries
- Utilizes libraries like `pandas`, `numpy`, `scipy`, `matplotlib`, and `seaborn` for data handling, statistical transformations, and visualizations.

### 2. Data Loading
- Loads a pre-processed dataset from a pickle file (`merged_p1_30k.pkl`).
- Sets the `Display Name` column as the index for filtering and manipulation.

### 3. Data Filtering
- Filters the dataset to include only selected wells (`B03`, `C03`, `D03`, etc.) relevant to the analysis.

### 4. Feature Transformation
- A predefined list of features (`FeatureNameList`) is processed:
  - Adds a constant value (`1`) to all selected features.
  - Applies log transformation to each feature.
  - Computes mean and standard deviation for normalization.

### 5. Data Extraction
- Defines a `DataExtractor` function that:
  - Loads well-specific feature data from `.npz` files.
  - Filters rows based on a feature area threshold.
  - Computes Z-scores for the selected features using the log-transformed mean and standard deviation.
  - Returns metadata (e.g., well name, Z-scores) for further aggregation.

### 6. Aggregation
- Iterates through a plate layout (rows: B-G, columns: 02-11) to extract and standardize feature data for each well.
- Aggregates results into lists for later compilation.

### 7. Output and Validation
- Flattens the aggregated data into a single list.
- Creates a DataFrame (`plate_3_zsn`) to store extracted and processed feature data.
- Prints key metrics (e.g., lengths of lists) to validate the extraction and processing steps.

---

## Applications
This script is part of a larger pipeline for analyzing high-content imaging data. The processed dataset can be used for:
- Statistical analysis.
- Visualization of cellular features.
- Machine learning or predictive modeling.

---

## Summary of Outputs
- Final processed DataFrame (`plate_3_zsn`) containing normalized feature data for selected wells.
- Debugging information, including lengths of aggregated lists, to ensure correctness.


In [ ]:
# Import necessary libraries
import pandas as pd  # For data manipulation
import pickle  # For loading and saving serialized data
import numpy as np  # For numerical computations
import matplotlib.pyplot as plt  # For visualization
from scipy.stats import norm, lognorm  # For statistical distribution calculations
import scipy.stats as stats  # For various statistical tools
import seaborn as sns  # For advanced visualizations

# Load pre-processed dataset from a pickle file
dfp = pd.read_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\merged_p1_30k.pkl')

# Set 'Display Name' column as the index for easy filtering and manipulation
dfp = dfp.set_index(['Display Name'])

# Filter rows based on specific well names (B03, C03, etc.)
dfp3 = dfp.loc[dfp.index.isin([
    'B03', 'C03', 'D03', 'E03', 'F03', 'G03'
])]

# Reset the index of the filtered dataset for further processing
dfp3.reset_index()

# Define a list of features for analysis
FeatureNameList = [
    'Region:Circularity', 'Region:Clumpiness', 'Region:Diameter, Mean',
    'Region:Fractal Dimension', 'Region:Heterogeneity', 'Region:Hole Area',
    'Region:Hole Area Ratio', 'Region:Roundness', 'Nuclear Area',
    'Percent Area Parent', 'Feature Area', 'red_intensity', 'green_intensity'
]

# Add a constant value (1) to all features in FeatureNameList
value_to_add = 1
dfp3.loc[:, FeatureNameList] += value_to_add

# Loop through each feature to apply transformations and compute statistics
for p in range(len(FeatureNameList)):
    # Extract the current feature
    dfp4 = dfp3[FeatureNameList[p]]

    # Apply log transformation to the feature
    dfp44 = np.log(dfp4)

    # Compute mean and standard deviation of the transformed feature
    mean = dfp44.mean()
    std = dfp44.std()

    # Define a function to extract and standardize data from specific wells
    def DataExtractor(Well, Feature=p):
        # Load well-specific data from an NPZ file
        Well_data_names = np.load("C:/Users/ak20adb/OneDrive - University of Hertfordshire/Desktop/Celleste 30k test/full datasets/plate_1/" + Well + ".npz")
        Well_data = Well_data_names['FeatureData']  # Feature data
        Well_names = Well_data_names['FeatureNames']  # Feature names

        # Filter data based on minimum feature area (not applied here)
        MinFeatureArea = 0
        GoodDataIndices = np.where(Well_data[:, 0])[0]
        Well_data = Well_data[GoodDataIndices, :]
        Well_names = Well_names[GoodDataIndices]

        # Standardize the selected feature by computing Z-scores
        Well_data[:, Feature] = Well_data[:, Feature] + 1
        zscore = (np.log(Well_data[:, Feature]) - mean) / std

        # Prepare DataFrame outputs for metadata and z-scores
        df['wells'] = pd.DataFrame([Well])
        df1['count'] = pd.DataFrame([len(Well_names)])
        df2 = pd.concat([df, df1], ignore_index=True)

        well_list = [Well]
        zscore_list = [zscore]

        return well_list, zscore_list, Well_names

    # Initialize lists for storing aggregated data
    appended_data = []
    appended_try = []
    appended_try2 = []
    appended_data2 = []
    append_lst = []

    # Iterate over rows (B to G) and columns (02 to 11) for well extraction
    letter = ['B', 'C', 'D', 'E', 'F', 'G']
    num = ['02', '03', '04', '05', '06', '07', '08', '09', '10', '11']

    for i in range(len(num)):
        for x in range(len(letter)):
            try:
                # Construct well name and extract data
                lst = str(letter[x]) + str(num[i])
                dfx1, dfx2, dfx3 = DataExtractor(lst)

                # Store results in lists
                appended_data.append(dfx1)
                appended_try.extend([lst] * len(dfx3))
                appended_data2.append(dfx2[0])
                append_lst.append(lst)

            except:
                # Handle cases where data extraction fails
                appended_data.append([lst])
                appended_try.append(lst)
                appended_data2.append({0})
                append_lst.append(lst)

    # Flatten the aggregated data into a single list
    flat_list = [item for sublist in appended_data2 for item in sublist]

    # Create a DataFrame for storing extracted data
    plate_3_zsn = pd.DataFrame(appended_try, columns=['Display Name'])

# Print summary statistics of the data
print("Length of flat_list:", len(flat_list))
print("Length of appended_try:", len(appended_try))
print("Length of appended_data2:", len(appended_data2))
print("Length of append_lst:", len(append_lst))

# Save or print the final DataFrame
plate_3_zsn.columns = ['Display Name']
print(plate_3_zsn)



## Purpose
This script processes cellular feature data extracted from high-content imaging experiments. It involves filtering, transforming, and standardizing feature data for specific wells, followed by aggregating and preparing the results for further analysis.

---

## Key Steps

### 1. **Import Libraries**
- **pandas**: For data manipulation and analysis.
- **pickle**: For loading serialized datasets.
- **numpy**: For numerical operations.
- **matplotlib**: For data visualization.
- **scipy.stats**: For statistical computations, including Z-scores and distributions.
- **seaborn**: For enhanced visualizations.

### 2. **Data Loading**
- Loads a pre-processed dataset from a pickle file (`p3 30k measurements.pkl`).
- Sets the `Display Name` column as the index for easier filtering and manipulation.

### 3. **Data Filtering**
- Filters the dataset to include specific wells (`B03`, `C03`, `D03`, etc.).
- Resets the index of the filtered dataset for further processing.

### 4. **Feature Transformation**
- Defines a list of features (`FeatureNameList`) to process, such as:
  - `Region:Area`
  - `Region:Circularity`
  - `Region:Clumpiness`
- Adds a constant value (`1`) to all selected features for standardization.
- Applies log transformation to normalize the data.

### 5. **Z-Score Standardization**
- Computes the **mean** and **standard deviation** of the log-transformed features.
- Standardizes feature values using Z-scores to allow comparison across wells.

### 6. **Data Extraction**
- Defines a function `DataExtractor` to process individual wells:
  - Loads well-specific feature data from `.npz` files.
  - Filters rows based on a feature area threshold (optional).
  - Normalizes features using Z-scores.
  - Returns well names, Z-scores, and processed feature data.

### 7. **Aggregation**
- Iterates through all combinations of rows (B-G) and columns (02-11) representing wells.
- Extracts and aggregates feature data for each well using the `DataExtractor` function.
- Flattens aggregated data into a single list for final processing.

### 8. **Final Output**
- Creates a DataFrame (`plate_3_zs`) containing processed feature data.
- Drops unnecessary columns (`Collect#`, `Feature Name`) for a cleaner dataset.
- Prints debugging information such as the lengths of aggregated lists and the final DataFrame.

---

## Applications
This script is part of a pipeline for analyzing high-content imaging data, enabling:
- Statistical analysis of cellular features.
- Normalization of data across multiple wells.
- Preparation of data for downstream analysis, such as machine learning or visualization.

---

## Summary of Outputs
1. **Processed DataFrame (`plate_3_zs`)**: Contains normalized and aggregated feature data for specific wells.
2. **Debugging Information**: Includes metrics like the length of aggregated lists for validation purposes.

Let me know if you need additional details or modifications to this summary!


In [ ]:
# Import necessary libraries
import pandas as pd  # For data manipulation
import pickle  # For saving/loading serialized data
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For data visualization
from scipy.stats import norm, lognorm  # For statistical distribution functions
import scipy.stats as stats  # For additional statistical computations
import seaborn as sns  # For advanced data visualizations

# Load a pre-processed dataset from a pickle file
dfp = pd.read_pickle(r'C:/Users/ak20adb/OneDrive - University of Hertfordshire/Desktop/p3 30k measurements.pkl')

# Set 'Display Name' as the index to allow easy filtering and manipulation of wells
dfp = dfp.set_index(['Display Name'])

# Filter the dataset for specific wells (e.g., B03, C03, etc.)
dfp3 = dfp.loc[dfp.index.isin([
    'B03', 'C03', 'D03', 'E03', 'F03', 'G03'
])]

# Reset the index of the filtered DataFrame for further processing
dfp3.reset_index()

# Define the list of features to process
FeatureNameList = [
    'Region:Area', 'Region:Circularity', 'Region:Clumpiness',
    'Region:Diameter, Mean', 'Region:Fractal Dimension',
    'Region:Heterogeneity', 'Region:Hole Area', 'Region:Hole Area Ratio',
    'Region:Percent Area Parent', 'Region:Roundness'
]

# Add a constant value (1) to the selected features for normalization
value_to_add = 1
dfp3.loc[:, FeatureNameList] += value_to_add

# Print the length of the filtered dataset and the feature list for debugging
print("Length of dfp3:", len(dfp3))
print("Length of FeatureNameList:", len(FeatureNameList))

# Loop through each feature for transformation and analysis
for p in range(len(FeatureNameList)):
    # Extract the feature column and apply log transformation
    dfp4 = dfp3[FeatureNameList[p]]
    dfp44 = np.log(dfp4)

    # Compute the mean and standard deviation of the transformed feature
    mean = dfp44.mean()
    std = dfp44.std()

    # Define a function to extract data for specific wells and compute Z-scores
    def DataExtractor(Well, Feature=p):
        # Load data from an NPZ file specific to the well
        Well_data_names = np.load("C:/Users/ak20adb/OneDrive - University of Hertfordshire/Desktop/test/bigp3/" + Well + ".npz")
        Well_data = Well_data_names['FeatureData']  # Feature values
        Well_names = Well_data_names['FeatureNames']  # Corresponding feature names

        # Filter data based on a minimum feature area threshold (currently not applied)
        GoodDataIndices = np.where(Well_data[:, 0])[0]
        Well_data = Well_data[GoodDataIndices, :]
        Well_names = Well_names[GoodDataIndices]

        # Standardize the feature using log transformation and Z-score normalization
        Well_data[:, Feature] += 1
        zscore = (np.log(Well_data[:, Feature]) - mean) / std

        # Prepare DataFrame outputs for metadata and Z-scores
        df['wells'] = pd.DataFrame([Well])
        df1['count'] = pd.DataFrame([len(Well_names)])
        df2 = pd.concat([df, df1], ignore_index=True)

        # Return the well name, Z-scores, and feature names
        return [Well], [zscore], Well_names

    # Initialize lists to store aggregated results
    appended_data = []
    appended_data2 = []
    append_lst = []
    appended_try = []
    letter = ['B', 'C', 'D', 'E', 'F', 'G']  # Well rows
    num = ['02', '03', '04', '05', '06', '07', '08', '09', '10', '11']  # Well columns

    # Iterate through all well combinations (rows and columns)
    for i in range(len(num)):
        for x in range(len(letter)):
            try:
                # Construct well name and extract data
                lst = str(letter[x]) + str(num[i])
                dfx1, dfx2, dfx3 = DataExtractor(lst)

                # Store results in respective lists
                appended_data.append(dfx1)
                appended_try.extend([lst] * len(dfx3))
                appended_data2.append(dfx2[0])
                append_lst.append(lst)

            except:
                # Handle missing data cases
                appended_data.append([lst])
                appended_data2.append({0})
                append_lst.append(lst)

    # Flatten the list of extracted data
    flat_list = [item for sublist in appended_data2 for item in sublist]

    # Prepare the final DataFrame for this feature
    plate_3_zs = dfp.iloc[:, [0, 1]].reset_index()
    plate_3_zs = plate_3_zs.drop(['Collect#', 'Feature Name'], axis=1)

    # Print debugging information for validation
    print("Length of flat_list:", len(flat_list))

# Print additional debugging information
print("Length of appended_try:", len(appended_try))
print("Length of appended_data2:", len(appended_data2))
print("Length of append_lst:", len(append_lst))

# Final output: Display the processed DataFrame
print(plate_3_zs)



### 2. **Load Dataset**
- The dataset is loaded from a pickle file (`plate_1_zs_new.pkl`) using `pandas.read_pickle()`.
- This file contains pre-processed data related to a specific plate.

### 3. **Add Metadata**
- A new column, `Plate_num`, is added to the dataset.
- The value `'plate_30k_1'` is assigned to all rows, uniquely identifying the plate associated with the data.

### 4. **Save Updated Dataset**
- The updated dataset, with the new `Plate_num` column, is saved back to the same pickle file, overwriting the original file.

### 5. **Debugging (Optional)**
- The script includes an optional print statement (`print(p3)`) to display the updated dataset for validation.

---

## Applications
- The script helps track the source of the data by adding metadata.
- Useful in workflows involving multiple plates or datasets where maintaining plate-specific information is critical.

---

## Outputs
- **Updated Pickle File**: The dataset now includes a `Plate_num` column for identifying the plate number (`plate_30k_1`).



In [ ]:
# Import necessary libraries
import numpy as np  # For numerical computations (not used in this snippet but likely for consistency in the workflow)
import pandas as pd  # For data manipulation and handling

# Load the dataset from a pickle file
# The file contains a dataset corresponding to "plate_1_zs_new.pkl"
p3 = pd.read_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_1_zs_new.pkl')

# Add a new column to the dataset to specify the plate number
# This ensures the data from this file can be uniquely identified as part of "plate_30k_1"
p3['Plate_num'] = 'plate_30k_1'

# Save the modified dataset back to the same pickle file
# This overwrites the original file with the updated version that now includes the 'Plate_num' column
p3.to_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_1_zs_new.pkl')

# Debugging step to print the updated DataFrame (this line is commented out in practice)
#print(p3)


merging of individually z scored plates for later analysis 

In [ ]:
#adding all the plates together

p3 = pd.read_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_1_zs_new.pkl')
p4 = pd.read_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_2_zs_new.pkl')
p5 = pd.read_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_3_zs_new.pkl')


p3 = pd.concat([p3,p4,p5],axis=0, ignore_index=True)

p3.to_pickle(r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_all_zs_new.pkl')
print(p3)